## Detect patients with multiple pathologies

In [58]:
import pandas as pd

In [59]:
healthy_data = pd.read_csv('data/healthy.csv')
healthy_data.dropna(inplace=True) #remove all NaN values

pathological_data = pd.read_csv('data/pathological.csv')
pathological_data.dropna(inplace=True) #remove all NaN values

In [60]:
#result_dict = pathological_data.groupby('person_id')['type'].apply(set).to_dict()
#filtered_dict = {k: v for k, v in result_dict.items() if len(v) > 1} #Filtered to only keep patients present in multiple pathologies

#print(len(result_dict))
#print(len(filtered_dict))
#print(filtered_dict)

## Binary Phrase Data

In [77]:
#Select only phrases
healthy_phrase = healthy_data[healthy_data['recording'] == 'phrase']
pathological_phrase = pathological_data[pathological_data['recording'] == 'phrase']
#print(len(healthy_phrase))
#print(len(pathological_phrase))

#Detect patients with multiple pathologies
result_dict = pathological_phrase.groupby('person_id')['type'].apply(set).to_dict()
filtered_dict = {k: v for k, v in result_dict.items() if len(v) > 1} #Filtered to only keep patients present in multiple pathologies

print(f"Number of patients: {len(result_dict)}")
print(f"Number of patients with multiple pathologies: {len(filtered_dict)}")
print("="*50)

#Removing duplicates (keeping the element within the most frequent pathology)

#Get pathology recording counts
pathology_recording_count = pathological_phrase['type'].value_counts()

#Initialize exclusion mask
exclusion_mask = pd.Series(False, index=pathological_phrase.index)

#Remove most frequent pathology from dictionary then update mask for later deduplication
for patient, pathologies in filtered_dict.items():
    #print(patient)
    most_frequent_pathology = max(pathologies, key=lambda item: pathology_recording_count.get(item, 0))
    pathologies.remove(most_frequent_pathology)
    for pathology in pathologies:
        #print(pathology+ "\n")
        exclusion_mask |= (pathological_phrase['type'] == pathology) & (pathological_phrase['person_id'] == patient)

#Remove duplicates
pathological_phrase = pathological_phrase[~exclusion_mask]

result_dict = pathological_phrase.groupby('person_id')['type'].apply(set).to_dict()
filtered_dict = {k: v for k, v in result_dict.items() if len(v) > 1} #Filtered to only keep patients present in multiple pathologies

print("After deduplication")
print(f"Number of patients: {len(result_dict)}")
print(f"Number of patients with multiple pathologies: {len(filtered_dict)}")


#Remove pathologies with less than 10 patients
print("="*50)
deduplicate_count = pathological_phrase.groupby('type')['person_id'].count().to_dict()
print(f"Number of pathologies before removal: {len(deduplicate_count)}")

for pathology, number in deduplicate_count.items():
    #print(pathology, number)
    if number<10:
        pathological_phrase.drop(pathological_phrase[pathological_phrase['type'] == pathology].index, inplace=True)

after_removal = pathological_phrase.groupby('type')['person_id'].count().to_dict()
print(f"Number of pathologies after removal: {len(after_removal)}")

#Now balance data

Number of patients: 1354
Number of patients with multiple pathologies: 262
After deduplication
Number of patients: 1354
Number of patients with multiple pathologies: 0
Number of pathologies before removal: 59
Number of pathologies after removal: 22


## Binary Vowel /a/ Data

## Binary Vowel /a/ at diffrent pitches Data

## Binary All vowels and pitches Data